# Cross-Validation Variance Analysis (Part 1/3) — SMI + Classical Models

**Authors:** Swagotam Malakar, Anamika Das, Dr. Ohidujjaman
**Environment:** Kaggle CPU (no GPU required)
**Estimated runtime:** ~100 minutes (SMI + 4 classical models × 5 seeds × 5 folds = 125 fits)

---

Part 1 of 3 — across-seed F1 variance for SMI + 4 classical TF-IDF baselines under 5×5-fold CV with seeds `[42, 123, 2024, 7, 99]`. Reports per-seed mean F1 and across-seed mean ± std.

See the **Kaggle Setup** cell below for required inputs and configuration.

## Kaggle Setup

Kaggle resets accelerator / internet / input settings after a notebook
re-import. Use this checklist before each run.

| Setting | Required value |
|---------|----------------|
| Accelerator | **None** (CPU only) |
| Internet | Off (not required) |
| Inputs | `Swarabyanjan_BEST_BALANCED_1to1.csv` (or `Swarabyanjan_Gold_Balanced_766.csv`) — gold-standard dataset (766 articles) |

**Add as Kaggle input:**
1. Right panel → **Add Input** → **Dataset** → search for the gold-standard
   dataset (e.g., `v18-human-gold-final` or `swarabyanjan`).
2. The notebook auto-discovers the CSV via `glob` over `/kaggle/input/**`,
   with local fallback paths for off-Kaggle development.

**Outputs (written to `/kaggle/working/`):**
- `cv_variance_smi_classical_results.json` — canonical per-seed + across-seed summary
- `cv_variance_smi_classical_results_table.csv` — flat results table
- `cv_variance_smi_classical_boxplot.png` — box plot of F1 across 5 seeds per model

No GPU, no internet, no HuggingFace model — pure CPU.


### 1. Environment Setup

In [1]:
# === 1. Environment Setup ===
import os, sys, time, json, warnings, glob, re, math, unicodedata, random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')  # headless backend (Kaggle commit mode)
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, matthews_corrcoef,
                             confusion_matrix, classification_report)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost not available, will skip XGBoost in classical CV.')

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"pandas {pd.__version__}, numpy {np.__version__}", flush=True)
print(f"matplotlib {matplotlib.__version__}, seaborn {sns.__version__}", flush=True)
print("NB15a: 5×5-fold CV variance analysis — SMI + 4 classical models (CPU only)", flush=True)


pandas 2.3.3, numpy 2.0.2
matplotlib 3.10.0, seaborn 0.13.2
NB15a: 5×5-fold CV variance analysis — SMI + 4 classical models (CPU only)


### 2. Configuration

In [2]:
# === 2. Configuration ===
SEEDS = [42, 123, 2024, 7, 99]
N_FOLDS = 5

# Reference single-seed F1 from master_comparison.csv (seed=42 run, NB1 + NB8).
# Used to check whether the single-seed result is representative of the 5-seed mean.
SINGLE_SEED_F1 = {
    'SMI':                   0.8091,   # NB8: 0.8091±0.0564 (5-fold CV, seed=42)
    'Random Forest':         0.7879,   # NB1: 0.7879±0.0146
    'Logistic Regression':   0.7771,   # NB1: 0.7771±0.0466
    'Linear SVM':            0.7636,   # NB1: 0.7636±0.0175
    'XGBoost':               0.7524,   # NB1: 0.7524±0.0126
}

OUTPUT_JSON = OUTPUT_DIR / 'cv_variance_smi_classical_results.json'
OUTPUT_CSV  = OUTPUT_DIR / 'cv_variance_smi_classical_results_table.csv'
OUTPUT_PNG  = OUTPUT_DIR / 'cv_variance_smi_classical_boxplot.png'

# Runtime estimates (for documentation in the printouts):
#   * SMI 5×5-fold CV:           ~1 min  (LogReg on 8 features × 25 fits)
#   * Classical 5×5-fold CV:     ~100 min (TF-IDF + 4 models × 5 seeds × 5 folds)

print(f"SEEDS = {SEEDS}")
print(f"N_FOLDS = {N_FOLDS}")
print(f"Total CV fits per model: {len(SEEDS) * N_FOLDS} (5 seeds × 5 folds)")
print(f"Reference single-seed (seed=42) F1 from master_comparison.csv:")
for k, v in SINGLE_SEED_F1.items():
    print(f"  {k:<24} F1 = {v:.4f}")


SEEDS = [42, 123, 2024, 7, 99]
N_FOLDS = 5
Total CV fits per model: 25 (5 seeds × 5 folds)
Reference single-seed (seed=42) F1 from master_comparison.csv:
  SMI                      F1 = 0.8091
  Random Forest            F1 = 0.7879
  Logistic Regression      F1 = 0.7771
  Linear SVM               F1 = 0.7636
  XGBoost                  F1 = 0.7524


### 3. Load Gold Standard

In [3]:
# === 3. Load Gold Standard (766 articles) ===
GOLD_FILENAME = 'Swarabyanjan_BEST_BALANCED_1to1.csv'
GOLD_FILENAME_CLEANED = 'Swarabyanjan_Gold_Balanced_766.csv'

def find_gold():
    """Search Kaggle input dirs + local paths for the gold-standard CSV."""
    for fname in [GOLD_FILENAME_CLEANED, GOLD_FILENAME]:
        # Kaggle input paths
        candidates = [
            f'/kaggle/input/v18-human-gold-final/{fname}',
            f'/kaggle/input/swarabyanjan/{fname}',
            f'/kaggle/input/{fname}',
        ]
        for c in candidates:
            if os.path.isfile(c):
                return c
        matches = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
        if matches:
            return matches[0]
        # Local paths (for development / testing)
        for local in [f'./{fname}', f'../data/{fname}', f'./data/{fname}',
                      f'/home/z/my-project/analysis/github_repo/data/{fname}',
                      f'/home/z/my-project/upload/{fname}']:
            if os.path.isfile(local):
                return local
    return GOLD_FILENAME  # let pd.read_csv raise a helpful error

GOLD_PATH = find_gold()
print(f'Gold CSV path: {GOLD_PATH}')

gold = pd.read_csv(GOLD_PATH)
print(f'Gold shape: {gold.shape}')
print(f'Columns: {list(gold.columns)}')

# Normalize column names (legacy schema uses news_source, cleaned uses corpus_batch)
if 'corpus_batch' not in gold.columns and 'news_source' in gold.columns:
    gold['corpus_batch'] = gold['news_source']
elif 'news_source' not in gold.columns and 'corpus_batch' in gold.columns:
    gold['news_source'] = gold['corpus_batch']

gold['headline'] = gold['headline'].fillna('').astype(str)
gold['body_text'] = gold['body_text'].fillna('').astype(str)
# Handle stub articles (body_text == "not_available") — replace with empty string.
gold.loc[gold['body_text'] == 'not_available', 'body_text'] = ''

# Build the unified text column for the classical TF-IDF models.
gold['text'] = gold['headline'] + ' ' + gold['body_text']

print(f'\nLabel distribution: {gold["best_label"].value_counts().to_dict()}')
print(f'Total: {len(gold)} | Yellow: {gold["best_label"].sum()} '
      f'| Non-yellow: {(gold["best_label"] == 0).sum()}')
print(f'\nText length stats:')
print(gold['text'].str.len().describe().round(0))
gold.head(3)


Gold CSV path: /kaggle/input/datasets/smalakarishere/swarabyanjan/Swarabyanjan_Gold_Balanced_766.csv
Gold shape: (766, 8)
Columns: ['article_id', 'headline', 'body_text', 'corpus_batch', 'article_length', 'best_label', 'best_confidence', 'best_note']

Label distribution: {0: 383, 1: 383}
Total: 766 | Yellow: 383 | Non-yellow: 383

Text length stats:
count     766.0
mean     1652.0
std      1195.0
min        20.0
25%       943.0
50%      1274.0
75%      1984.0
max      8397.0
Name: text, dtype: float64


,article_id,headline,body_text,corpus_batch,article_length,best_label,best_confidence,best_note,news_source,text
0,v18_2830,কুমিল্লায় ‘ডাকাতের গুলিতে ডাকাত’ নিহত,"দাউদকান্দিথানার ওসি মিজানুর রহমান বলেন, “দুই দ...",corpus_expansion_5000,694,0,H,Crime news with named Daudkandi OC Mizanur Rah...,corpus_expansion_5000,কুমিল্লায় ‘ডাকাতের গুলিতে ডাকাত’ নিহত দাউদকান...
1,v18_0000,২০১৮ সালে বিশ্বজুড়ে ৯৭ সাংবাদিক খুন,বিশ্বজুড়ে সাংবাদিকদের ওপর হামলার ঘটনাগুলোতে ব...,new_low_w0_pany,1205,0,H,Factual international press-freedom report wit...,new_low_w0_pany,২০১৮ সালে বিশ্বজুড়ে ৯৭ সাংবাদিক খুন বিশ্বজুড়...
2,v18_4780,বিশ্বকাপে আফগানদের বিপক্ষে টেস্ট খেললেন ধোনি?,খেলা শুরুর আগেই 'ফেভারিট বনাম লাস্টবয়' শিরোনা...,corpus_expansion_5000,967,1,M,[R1-BROADER] Clickbait sports headline ('?'); ...,corpus_expansion_5000,বিশ্বকাপে আফগানদের বিপক্ষে টেস্ট খেললেন ধোনি? ...


### 4. SMI Scoring Functions

In [4]:
# === 4. SMI Scoring Functions ===
# Copied EXACTLY from NB8_SMI_Annotation_Experiment.ipynb (cell 3).
# These implement the mathematical definitions C1-C8 from the paper.

import re
import math
import unicodedata

# --- Lexicons ---

SENSATIONAL_HEADLINE_TERMS = [
    "অবিশ্বাস্য", "অকল্পনীয়", "চমকে", "চাঞ্চল্যকর", "রোমহর্ষক",
    "ভয়ঙ্কর", "নারকীয়", "মর্মান্তিক", "বিভীষিকাময়",
    "চরম", "মহা", "প্রচণ্ড", "কেলেঙ্কারি", "কেলো", "হয়রানি",
    "আলোচিত", "বিতর্কিত", "রহস্যময়", "রহস্য",
    "তবে কি", "তবে কী", "কী ঘটল", "কী হলো",
    "রহস্যের", "রহস্য জট", "জট খুলল", "পর্দা ফাঁক",
    "অবাক", "হতবাক", "স্তব্ধ", "বিস্ময়ে হতবাক",
    "কাঁদছে", "ফাটল", "ছিন্নভিন্ন", "তোলপাড়", "নড়েচড়ে",
    "চাঞ্চল্য", "শিহরণ", "আঁতকে", "কাঁপিয়ে", "কাঁপছে",
]

CLICKBAIT_PHRASES = [
    "তবে কি", "তবে কী", "জানলে অবাক", "যা ঘটল", "যা কেউ বলেনি",
    "ভাবেননি", "অবাক করবে", "চমকে দেওয়া", "অজানা সত্য",
    "এক চমকে", "হয়তো ভাবেননি", "যা দেখলে", "বিশ্বাস করবেন না",
    "নিজের চোখে দেখুন", "ভিডিওতে দেখুন", "ছবিতে দেখুন",
    "পুরো ঘটনা", "পুরো রহস্য", "না জানলে মিস", "অপেক্ষা করুন",
    "রহস্যের জট", "মজার", "মজার তথ্য",
    "যা আপনি জানেন না", "গোপন তথ্য", "আসল সত্য",
    "চমকপ্রদ", "নজরকাড়া", "অভাবনীয়",
    "অবশ্যই দেখুন", "শেয়ার করুন", "ভাইরাল",
    "দেখে নিন", "জেনে নিন", "চিনে নিন",
    "বিস্ময়কর", "অকল্পনীয়", "অবিশ্বাস্য",
]

CLICKBAIT_LISTICLE_RE = re.compile(
    r"(\d+|১|২|৩|৪|৫|৬|৭|৮|৯|১০)\s*(টি|টা|ভাবে|কারণে|টিপস|পদ্ধতি|উপায়)"
)

EMOTIONAL_TERMS = [
    "অশ্রু", "কান্না", "হাহাকার", "বিলাপ", "করুণ", "করুণতা",
    "কান্নায় ভেঙে", "শোকে", "শোকাহত", "বিলাপ করছেন",
    "করুণ আর্তনাদ", "আর্তনাদ", "হাহাকার শুরু",
    "বুক ফেটে", "হৃদয় বিদারণ", "মর্মান্তিক", "নারকীয়",
    "বিভীষিকাময়", "রোমহর্ষক", "কম্পিত", "কাঁপছে",
    "হাহাকারে", "হাহাকার উঠেছে", "রোদন",
    "বিষণ্ণ", "হতাশ", "হতাশা", "নিরাশা",
    "উল্লাসে", "উল্লাসিত", "আনন্দে", "আনন্দঘন",
    "ক্ষোভে", "ক্ষুব্ধ", "রুষ্ট", "ক্ষোভ প্রকাশ",
    "বিক্ষোভ", "ধিক্কার", "নিন্দা", "প্রতিবাদ",
]

ATTRIBUTION_TERMS = [
    "বলেন", "জানিয়েছেন", "জানান", "বলা হয়েছে", "বলেছেন",
    "মতে", "অনুসারে", "সূত্রে", "সূত্র বলছে",
    "নিশ্চিত করেছেন", "নিশ্চিত করা হয়েছে",
    "প্রকাশ করেছেন", "প্রকাশ করেছে",
    "জানিয়েছে", "বলা হয়", "যোগ করেছেন",
    "রইটার্স", "রয়টার্স", "রয়টার", "বিডিনিউজ", "বাসস", "ইউএনবি",
    "এএফপি", "এপি", "ডিপিএ",
    "প্রতিবেদক", "প্রতিনিধি", "নিজস্ব প্রতিবেদক",
    "সংস্থা", "সংস্দা", "সংবাদ সংস্থা",
    "বিবৃতি", "প্রেস বিবৃতি", "বিজ্ঞপ্তি", "প্রেস রিলিজ",
    "আদালত", "পুলিশ", "মন্ত্রণালয়", "সরকার", "সংসদ",
    "বিভাগ", "অধিদপ্তর", "পরিষদ", "কমিটি", "কমিশন",
    "টিআইবি", "ট্রান্সপারেন্সি ইন্টারন্যাশনাল",
    "রিপোর্ট", "প্রতিবেদন", "তদন্ত", "অনুসন্ধান",
    "বিশেষজ্ঞ", "বিশ্লেষক", "অধ্যাপক", "ডাক্তার",
    "মামলা", "রায়", "আদেশ", "নোটিশ",
]

SPECULATION_TERMS = [
    "হতে পারে", "হতে পারেন", "থাকতে পারে", "হয়তো", "সম্ভবত",
    "মনে হচ্ছে", "মনে হয়", "অনুমান", "গুঞ্জন", "গুঞ্জন রটে",
    "সম্ভাবনা", "সম্ভব", "সম্ভাব্য",
    "জল্পনা", "কল্পনা", "জল্পনা-কল্পনা",
    "নাকি", "কি তবে", "তবে কি", "তবে কী",
    "শোনা যাচ্ছে", "জানা গেছে যে", "খবর রটে",
    "চর্চা শুরু", "বিতর্ক শুরু", "প্রশ্ন উঠেছে",
]

ENTERTAINMENT_TERMS = [
    "অভিনেত্রী", "অভিনেতা", "মডেল", "গায়ক", "গায়িকা", "নায়ক", "নায়িকা",
    "বলিউড", "হলিউড", "টলিউড", "ঢালিউড",
    "ব্যক্তিগত জীবন", "প্রেম", "প্রেমের", "বিবাহবিচ্ছেদ",
    "ছাড়াছাড়ি", "বিয়ে", "বিয়ের", "প্রেমের গল্প", "নতুন জুটি",
    "ভাইরাল", "টুইট", "ইনস্টাগ্রামে",
    "ছবি ভাইরাল", "ভিডিও ভাইরাল", "ছবি ফাঁস", "অন্তরঙ্গ",
    "চলচ্চিত্র", "প্রিমিয়ার", "শুটিং", "সিনেমা", "নাটক",
    "অভিনয়", "মুক্তি", "বক্স অফিস", "ট্রেইলর",
    "গসিপ", "ফটোশুট", "মেকআপ", "ড্রেস", "গাউন",
    "বিউটি", "ফিটনেস", "ওজন কমানো", "ফিগার", "সাইজ জিরো",
    "পুরস্কার", "এওয়ার্ড", "অস্কার",
]

SENSITIVE_TOPIC_TERMS = [
    # Communal / religious
    "মুসলমান", "হিন্দু", "ইসলাম", "হিন্দুধর্ম", "মন্দির", "মসজিদ", "মাদ্রাসা",
    "ধর্মীয়", "ধর্ম", "সাম্প্রদায়িক", "সম্প্রদায়িক", "দাঙ্গা", "দাঙ্গাহাঙ্গামা",
    "উসকানি", "উসকানি দিয়েছে", "ধর্মান্ধ", "কট্টর", "অমুসলিম", "কাফির",
    # Gender / sexual
    "ধর্ষণ", "ধর্ষিতা", "নারী নির্যাতন", "যৌন হয়রানি", "ইভ টিজিং",
    "নারীবাদী", "মেয়েদের", "নারীদের অধিকার",
    # Ethnicity / regional
    "উপজাতি", "চাকমা", "মারমা", "ত্রিপুরা", "গারো", "সাঁওতাল",
    "আদিবাসী", "পাহাড়ি", "সমতট",
    # Political provocation
    "সরকারবিরোধী", "বিরোধীদল", "ক্ষমতাসীন", "আওয়ামী লীগ", "বিএনপি",
    "জামায়াত", "জাতীয় পার্টি", "হেফাজত", "ছাত্রলীগ", "ছাত্রদল",
    "জিহাদ", "শহীদ", "শহীদের", "রাজাকার", "আলবদর",
    "বয়কট", "অবরোধ", "অচলাবস্থা", "ধর্মঘট",
    "বিচ্ছিন্নতাবাদী", "স্বাধীনতাবিরোধী",
]

BENGALI_STOPWORDS = {
    "এবং", "ও", "এর", "কে", "কেও", "তিনি", "তার", "তাকে", "তাদের",
    "এই", "সেই", "ঐ", "এক", "একটি", "একটা", "একজন",
    "হয়েছে", "হয়েছিল", "হবে", "হতে", "করেছেন", "করেছে",
    "বলেন", "বলেছেন", "যিনি", "যে", "যা",
    "আজ", "গতকাল", "আগামীকাল",
    "তবে", "কিন্তু", "আর", "অথচ", "যদিও",
    "কারণ", "তাই", "সুতরাং",
    "নিয়ে", "দিয়ে", "থেকে", "ভিতরে", "বাইরে",
    "সাথে", "সঙ্গে", "নিচে", "উপরে",
    "সব", "অনেক", "কিছু", "কোনো", "অন্য", "নিজে",
}

DATELINE_RE = re.compile(
    r"^[^\s,]{2,15}\s*,\s*[\d০-৯]|^[^\s]{2,15}\s*\([^)]+\)\s*[-—]"
)

# --- Helper functions ---

def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u200d", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def count_term_hits(text, terms):
    if not text:
        return 0
    return sum(1 for t in terms if t in text)

def count_total_term_hits(text, terms):
    if not text:
        return 0
    return sum(text.count(t) for t in terms)

def word_count(text):
    if not text:
        return 0
    return len(text.split())

def has_strong_attribution(headline, body):
    full = normalize_text(headline or "") + " " + normalize_text(body or "")
    credible_sources = [
        "টিআইবি", "ট্রান্সপারেন্সি", "রয়টার্স", "রইটার্স", "বিডিনিউজ",
        "বাসস", "ইউএনবি", "এএফপি", "বিশ্বব্যাংক", "আইএমএফ",
        "জাতিসংঘ", "ইউনিসেফ", "বিশ্ববিদ্যালয়", "গবেষণা", "সমীক্ষা",
        "আদালত", "পুলিশ", "র‌্যাব", "সিআইডি", "মন্ত্রণালয়",
        "প্রতিবেদক", "প্রতিনিধি", "নিজস্ব প্রতিবেদক",
        "বিজ্ঞপ্তি", "বিবৃতি",
    ]
    return any(src in full for src in credible_sources)

def has_dateline(body):
    b = normalize_text(body or "")[:200]
    return bool(DATELINE_RE.match(b))

# --- Eight Criteria Scoring Functions ---

def C1_sensational_headline(headline):
    """C1: Sensational headline score in [0,1]."""
    if not headline:
        return 0.0
    h = normalize_text(headline)
    hits = count_term_hits(h, SENSATIONAL_HEADLINE_TERMS)
    marks = h.count("!") + h.count("?")
    base = min(hits / 2.0, 1.0)
    mark_bonus = min(marks / 1.5, 0.3)
    return min(base + mark_bonus, 1.0)

def C2_clickbait(headline, body):
    """C2: Clickbait score in [0,1]."""
    h = normalize_text(headline or "")
    phrase_hits = count_term_hits(h, CLICKBAIT_PHRASES)
    listicle_hit = 1 if CLICKBAIT_LISTICLE_RE.search(h) else 0
    trailing_q = 1 if (h.endswith("?") or h.endswith("…") or h.endswith("...")) else 0
    base = min(phrase_hits / 1.5, 1.0)
    bonus = 0.15 * listicle_hit + 0.20 * trailing_q
    return min(base + bonus, 1.0)

def C3_emotional(body):
    """C3: Emotional arousal score in [0,1].
    Formula: 1 - exp(-D/gamma), D = density per 100 words
    """
    b = normalize_text(body or "")
    if not b:
        return 0.0
    wc = word_count(b)
    if wc == 0:
        return 0.0
    hits = count_total_term_hits(b, EMOTIONAL_TERMS)
    density = hits / max(wc / 100.0, 1.0)
    gamma = 1.2
    score = 1 - math.exp(-density / gamma)
    return min(score, 1.0)

def C4_attribution_gap(headline, body):
    """C4: Attribution gap score in [0,1].
    Formula: max(1 - lambda*n_attr - credits, 0) + short_penalty
    """
    b = normalize_text(body or "")
    h = normalize_text(headline or "")
    full = h + " " + b
    wc = word_count(b)
    if wc == 0:
        return 1.0
    attr_hits = count_term_hits(full, ATTRIBUTION_TERMS)
    has_strong = has_strong_attribution(h, b)
    has_dl = has_dateline(b)
    lam = 0.10
    base = max(1.0 - lam * attr_hits, 0.0)
    if has_strong:
        base = max(base - 0.30, 0.0)
    if has_dl:
        base = max(base - 0.15, 0.0)
    if wc < 100:
        base = min(base + 0.05, 1.0)
    return min(max(base, 0.0), 1.0)

def C5_speculation(body):
    """C5: Speculation-as-fact score in [0,1].
    Formula: 1 - exp(-D/gamma)
    """
    b = normalize_text(body or "")
    if not b:
        return 0.0
    wc = word_count(b)
    hits = count_total_term_hits(b, SPECULATION_TERMS)
    if wc == 0:
        return 0.0
    density = hits / max(wc / 100.0, 1.0)
    gamma = 1.2
    score = 1 - math.exp(-density / gamma)
    return min(score, 1.0)

def C6_entertainment(headline, body):
    """C6: Entertainment displacement score in [0,1].
    Formula: min(hits/alpha + 0.25*headline_hits, 1)
    """
    h = normalize_text(headline or "")
    b = normalize_text(body or "")
    full = h + " " + b
    hits = count_term_hits(full, ENTERTAINMENT_TERMS)
    headline_hits = count_term_hits(h, ENTERTAINMENT_TERMS)
    alpha = 3.0
    base = min(hits / alpha, 1.0)
    headline_bonus = min(0.25 * headline_hits, 0.5)
    return min(base + headline_bonus, 1.0)

def C7_coherence(headline, body):
    """C7: Headline-body coherence (mismatch) score in [0,1].
    Formula: 1 - overlap_ratio if overlap < tau, else 0
    """
    h = normalize_text(headline or "")
    b = normalize_text(body or "")
    if not h or not b:
        return 0.3
    h_tokens = set(re.findall(r"[\u0980-\u09FF]+|[A-Za-z]+|\d+", h))
    b_tokens = set(re.findall(r"[\u0980-\u09FF]+|[A-Za-z]+|\d+", b))
    h_tokens = {t for t in h_tokens if len(t) > 1 and t not in BENGALI_STOPWORDS}
    b_tokens = {t for t in b_tokens if len(t) > 1 and t not in BENGALI_STOPWORDS}
    if not h_tokens:
        return 0.3
    overlap = h_tokens & b_tokens
    overlap_ratio = len(overlap) / len(h_tokens)
    tau = 0.35
    if overlap_ratio < tau:
        return 1.0 - overlap_ratio
    return 0.0

def C8_sensitive_topic(headline, body):
    """C8: Sensitive topic score in [0,1].
    Formula: 1 - exp(-D/gamma), D = density per 100 words on
    combined headline+body, gamma = 1.5.
    """
    h = normalize_text(headline or "")
    b = normalize_text(body or "")
    full = h + " " + b
    if not full.strip():
        return 0.0
    wc = word_count(full)
    if wc == 0:
        return 0.0
    hits = count_total_term_hits(full, SENSITIVE_TOPIC_TERMS)
    density = hits / max(wc / 100.0, 1.0)
    gamma = 1.5
    score = 1 - math.exp(-density / gamma)
    return min(score, 1.0)

def compute_all_criteria(headline, body):
    """Compute all 8 criteria scores for an article."""
    return {
        'C1': round(C1_sensational_headline(headline), 4),
        'C2': round(C2_clickbait(headline, body), 4),
        'C3': round(C3_emotional(body), 4),
        'C4': round(C4_attribution_gap(headline, body), 4),
        'C5': round(C5_speculation(body), 4),
        'C6': round(C6_entertainment(headline, body), 4),
        'C7': round(C7_coherence(headline, body), 4),
        'C8': round(C8_sensitive_topic(headline, body), 4),
    }

print('SMI criteria scoring functions defined (copied EXACTLY from NB8).')
print(f'  C1: Sensational Headline (lexicon size: {len(SENSATIONAL_HEADLINE_TERMS)})')
print(f'  C2: Clickbait (lexicon size: {len(CLICKBAIT_PHRASES)})')
print(f'  C3: Emotional Arousal (lexicon size: {len(EMOTIONAL_TERMS)})')
print(f'  C4: Attribution Gap (lexicon size: {len(ATTRIBUTION_TERMS)})')
print(f'  C5: Speculation (lexicon size: {len(SPECULATION_TERMS)})')
print(f'  C6: Entertainment (lexicon size: {len(ENTERTAINMENT_TERMS)})')
print(f'  C7: Headline-Body Coherence')
print(f'  C8: Sensitive Topic (lexicon size: {len(SENSITIVE_TOPIC_TERMS)})')


SMI criteria scoring functions defined (copied EXACTLY from NB8).
  C1: Sensational Headline (lexicon size: 41)
  C2: Clickbait (lexicon size: 38)
  C3: Emotional Arousal (lexicon size: 40)
  C4: Attribution Gap (lexicon size: 59)
  C5: Speculation (lexicon size: 26)
  C6: Entertainment (lexicon size: 49)
  C7: Headline-Body Coherence
  C8: Sensitive Topic (lexicon size: 57)


### 5. Compute SMI Features

In [5]:
# === 5. Compute SMI Features Once (C1-C8 for all 766 articles) ===
# We compute the 8 SMI criteria ONCE per article and reuse the feature matrix
# across all 5 seeds. (The SMI criteria are deterministic functions of the
# text — they do not depend on the CV seed.)

print('Computing SMI criteria scores for 766 gold-standard articles (ONE TIME)...')
t0 = time.time()

criteria_rows = []
for _, row in gold.iterrows():
    c = compute_all_criteria(row['headline'], row['body_text'])
    c['article_id'] = row['article_id']
    c['true_label'] = int(row['best_label'])
    criteria_rows.append(c)

gold_criteria = pd.DataFrame(criteria_rows)
t1 = time.time()
print(f'Done in {t1-t0:.2f}s  —  shape: {gold_criteria.shape}')

# Feature matrix (n=766, d=8) and label vector (n=766,)
X_smi = gold_criteria[['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8']].values
y_smi = gold_criteria['true_label'].values

print(f'\nSMI feature matrix: X_smi.shape = {X_smi.shape}, y_smi.shape = {y_smi.shape}')
print(f'\nCriteria score means by label:')
for c in ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8']:
    m_y = gold_criteria[gold_criteria.true_label == 1][c].mean()
    m_n = gold_criteria[gold_criteria.true_label == 0][c].mean()
    print(f'  {c}: Yellow={m_y:.3f}, Non-yellow={m_n:.3f}, Diff={m_y-m_n:+.3f}')

gold_criteria.head()


Computing SMI criteria scores for 766 gold-standard articles (ONE TIME)...
Done in 2.36s  —  shape: (766, 10)

SMI feature matrix: X_smi.shape = (766, 8), y_smi.shape = (766,)

Criteria score means by label:
  C1: Yellow=0.186, Non-yellow=0.016, Diff=+0.171
  C2: Yellow=0.029, Non-yellow=0.002, Diff=+0.027
  C3: Yellow=0.060, Non-yellow=0.050, Diff=+0.010
  C4: Yellow=0.598, Non-yellow=0.361, Diff=+0.237
  C5: Yellow=0.202, Non-yellow=0.101, Diff=+0.102
  C6: Yellow=0.381, Non-yellow=0.092, Diff=+0.289
  C7: Yellow=0.136, Non-yellow=0.092, Diff=+0.044
  C8: Yellow=0.167, Non-yellow=0.239, Diff=-0.072


,C1,C2,C3,C4,C5,C6,C7,C8,article_id,true_label
0,0.0,0.0,0.0000,0.25,0.0,0.0000,0.0,0.9619,v18_2830,0
1,0.0,0.0,0.0000,0.20,0.0,0.0000,0.0,0.0000,v18_0000,0
2,0.3,0.2,0.0000,1.00,0.0,0.3333,0.0,0.0000,v18_4780,1
3,0.0,0.0,0.0000,0.20,0.0,1.0000,0.0,0.6832,v18_0416,0
4,0.0,0.0,0.4606,0.90,0.0,0.0000,0.0,0.0000,v18_4000,1


### 6. SMI 5×5-Fold CV

In [6]:
# === 6. Run 5×5-Fold CV for SMI ===
# For each of the 5 seeds, create a fresh StratifiedKFold(n_splits=5, shuffle=True,
# random_state=seed) and train LogisticRegression on the 8 SMI features.
# We record per-fold F1 (5 folds per seed) AND per-run mean F1 (1 number per seed).

print('=' * 70)
print('SMI 5×5-Fold CV  (5 seeds × 5 folds = 25 fits, LogReg on 8 features)')
print('=' * 70)

smi_per_seed_results = {}  # seed -> {'per_fold_f1': [...], 'run_mean_f1': float, 'run_std_f1': float}

for seed_i, seed in enumerate(SEEDS):
    print(f'\n--- SMI Seed {seed} ({seed_i+1}/{len(SEEDS)}) ---')
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    per_fold_f1 = []
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(X_smi, y_smi)):
        lr = LogisticRegression(
            C=1.0, max_iter=2000, class_weight='balanced', random_state=seed
        )
        lr.fit(X_smi[train_idx], y_smi[train_idx])
        preds = lr.predict(X_smi[val_idx])
        f1 = f1_score(y_smi[val_idx], preds)
        per_fold_f1.append(float(f1))
        print(f'  Seed {seed}  Fold {fold_i+1}/{N_FOLDS}: F1={f1:.4f}')
    run_mean = float(np.mean(per_fold_f1))
    run_std = float(np.std(per_fold_f1))
    smi_per_seed_results[seed] = {
        'per_fold_f1': per_fold_f1,
        'run_mean_f1': run_mean,
        'run_std_f1': run_std,
    }
    print(f'  -> Seed {seed} mean F1 = {run_mean:.4f} ± {run_std:.4f}')

# Across-seed summary
smi_per_seed_f1 = [smi_per_seed_results[s]['run_mean_f1'] for s in SEEDS]
smi_mean = float(np.mean(smi_per_seed_f1))
smi_std = float(np.std(smi_per_seed_f1))
print(f'\n>>> SMI across 5 seeds: F1 = {smi_mean:.4f} ± {smi_std:.4f}')
print(f'    Per-seed means: {[round(f, 4) for f in smi_per_seed_f1]}')
print(f'    Single-seed (seed=42) reference: {SINGLE_SEED_F1["SMI"]:.4f}')


SMI 5×5-Fold CV  (5 seeds × 5 folds = 25 fits, LogReg on 8 features)

--- SMI Seed 42 (1/5) ---
  Seed 42  Fold 1/5: F1=0.7361
  Seed 42  Fold 2/5: F1=0.9079
  Seed 42  Fold 3/5: F1=0.8163
  Seed 42  Fold 4/5: F1=0.7826
  Seed 42  Fold 5/5: F1=0.8027
  -> Seed 42 mean F1 = 0.8091 ± 0.0564

--- SMI Seed 123 (2/5) ---
  Seed 123  Fold 1/5: F1=0.8356
  Seed 123  Fold 2/5: F1=0.7914
  Seed 123  Fold 3/5: F1=0.8194
  Seed 123  Fold 4/5: F1=0.7703
  Seed 123  Fold 5/5: F1=0.8188
  -> Seed 123 mean F1 = 0.8071 ± 0.0233

--- SMI Seed 2024 (3/5) ---
  Seed 2024  Fold 1/5: F1=0.8105
  Seed 2024  Fold 2/5: F1=0.7724
  Seed 2024  Fold 3/5: F1=0.8243
  Seed 2024  Fold 4/5: F1=0.7681
  Seed 2024  Fold 5/5: F1=0.8286
  -> Seed 2024 mean F1 = 0.8008 ± 0.0257

--- SMI Seed 7 (4/5) ---
  Seed 7  Fold 1/5: F1=0.8299
  Seed 7  Fold 2/5: F1=0.8112
  Seed 7  Fold 3/5: F1=0.7945
  Seed 7  Fold 4/5: F1=0.7972
  Seed 7  Fold 5/5: F1=0.8392
  -> Seed 7 mean F1 = 0.8144 ± 0.0176

--- SMI Seed 99 (5/5) ---
  Seed

### 7. Classical 5×5-Fold CV

In [7]:
# === 7. Run 5×5-Fold CV for Classical Models ===
# For each seed, for each classical model (LogReg, RandomForest, LinearSVM,
# XGBoost): TF-IDF (fit on train fold only) + 5-fold CV.
# Estimated runtime: ~5 min per model per seed × 4 models × 5 seeds ≈ 100 min total.
# Hyperparameters copied EXACTLY from NB1_BanglaBERT_Classical.ipynb (cell 5).

print('=' * 70)
print('CLASSICAL 5×5-Fold CV  (5 seeds × 5 folds × 4 models = 100 fits)')
print('  Expected runtime: ~100 min on Kaggle CPU')
print('=' * 70)

X_text = gold['text'].values
y = gold['best_label'].values


def make_classical_models(seed):
    """Return a dict of name -> factory for the 4 classical models.

    Hyperparameters copied EXACTLY from NB1 (cell 5).
    """
    models = {
        'Logistic Regression': lambda: LogisticRegression(
            max_iter=1000, C=1.0, solver='lbfgs',
            class_weight='balanced', random_state=seed
        ),
        'Random Forest': lambda: RandomForestClassifier(
            n_estimators=200, max_depth=None,
            class_weight='balanced', random_state=seed, n_jobs=-1
        ),
        'Linear SVM': lambda: CalibratedClassifierCV(
            LinearSVC(C=1.0, class_weight='balanced', max_iter=2000,
                      random_state=seed),
            cv=3, method='sigmoid'
        ),
    }
    if HAS_XGB:
        models['XGBoost'] = lambda: XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            random_state=seed, n_jobs=-1, eval_metric='logloss',
            use_label_encoder=False
        )
    return models


classical_per_seed_results = {}  # model_name -> { seed -> {'per_fold_f1':..., 'run_mean_f1':...} }

t_total_start = time.time()
for seed_i, seed in enumerate(SEEDS):
    print(f'\n--- Classical Seed {seed} ({seed_i+1}/{len(SEEDS)}) ---')
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    folds = list(skf.split(X_text, y))
    models = make_classical_models(seed)

    for model_name, factory in models.items():
        if model_name not in classical_per_seed_results:
            classical_per_seed_results[model_name] = {}
        per_fold_f1 = []
        t_model_start = time.time()
        for fold_i, (train_idx, val_idx) in enumerate(folds):
            # TF-IDF fit on TRAIN fold only (no leakage)
            tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
            X_train = tfidf.fit_transform(X_text[train_idx])
            X_val = tfidf.transform(X_text[val_idx])

            model = factory()
            model.fit(X_train, y[train_idx])
            preds = model.predict(X_val)
            f1 = f1_score(y[val_idx], preds)
            per_fold_f1.append(float(f1))

        run_mean = float(np.mean(per_fold_f1))
        run_std = float(np.std(per_fold_f1))
        classical_per_seed_results[model_name][seed] = {
            'per_fold_f1': per_fold_f1,
            'run_mean_f1': run_mean,
            'run_std_f1': run_std,
        }
        elapsed = time.time() - t_model_start
        print(f'  {model_name:<24} Seed {seed}: F1={run_mean:.4f} ± {run_std:.4f}  ({elapsed:.1f}s)')

t_total = time.time() - t_total_start
print(f'\nTotal classical CV wall-time: {t_total/60:.1f} min')

# Across-seed summary for each classical model
print('\n--- Across-seed summary (5 seeds) ---')
for model_name in classical_per_seed_results:
    per_seed_f1 = [classical_per_seed_results[model_name][s]['run_mean_f1'] for s in SEEDS]
    mean = float(np.mean(per_seed_f1))
    std = float(np.std(per_seed_f1))
    ref = SINGLE_SEED_F1.get(model_name, None)
    ref_str = f'  (ref seed=42: {ref:.4f})' if ref is not None else ''
    print(f'  {model_name:<24} F1 = {mean:.4f} ± {std:.4f}{ref_str}')


CLASSICAL 5×5-Fold CV  (5 seeds × 5 folds × 4 models = 100 fits)
  Expected runtime: ~100 min on Kaggle CPU

--- Classical Seed 42 (1/5) ---
  Logistic Regression      Seed 42: F1=0.7794 ± 0.0405  (2.7s)
  Random Forest            Seed 42: F1=0.7698 ± 0.0105  (7.0s)
  Linear SVM               Seed 42: F1=0.7704 ± 0.0251  (2.7s)
  XGBoost                  Seed 42: F1=0.7549 ± 0.0168  (26.3s)

--- Classical Seed 123 (2/5) ---
  Logistic Regression      Seed 123: F1=0.7876 ± 0.0330  (2.5s)
  Random Forest            Seed 123: F1=0.7920 ± 0.0543  (6.8s)
  Linear SVM               Seed 123: F1=0.7804 ± 0.0355  (2.7s)
  XGBoost                  Seed 123: F1=0.7473 ± 0.0339  (26.4s)

--- Classical Seed 2024 (3/5) ---
  Logistic Regression      Seed 2024: F1=0.7796 ± 0.0426  (2.6s)
  Random Forest            Seed 2024: F1=0.7848 ± 0.0339  (6.9s)
  Linear SVM               Seed 2024: F1=0.7676 ± 0.0521  (2.6s)
  XGBoost                  Seed 2024: F1=0.7222 ± 0.0455  (26.5s)

--- Classical Seed

### 8. Results Table

In [8]:
# === 8. Results Table ===
# Build the master results table showing per-seed F1 (5 seeds) + across-seed
# mean ± std for SMI and the 4 classical models.

rows = []

# --- SMI ---
rows.append({
    'Model': 'SMI',
    **{f'Seed {s} F1': round(smi_per_seed_results[s]['run_mean_f1'], 4) for s in SEEDS},
    'Mean ± Std': f'{smi_mean:.4f} ± {smi_std:.4f}',
})

# --- Classical models (preserve NB1's master_comparison.csv ordering) ---
classical_order = ['Random Forest', 'Logistic Regression', 'Linear SVM', 'XGBoost']
for model_name in classical_order:
    if model_name not in classical_per_seed_results:
        continue
    per_seed_f1 = [classical_per_seed_results[model_name][s]['run_mean_f1'] for s in SEEDS]
    mean = float(np.mean(per_seed_f1))
    std = float(np.std(per_seed_f1))
    rows.append({
        'Model': model_name,
        **{f'Seed {s} F1': round(classical_per_seed_results[model_name][s]['run_mean_f1'], 4) for s in SEEDS},
        'Mean ± Std': f'{mean:.4f} ± {std:.4f}',
    })

results_df = pd.DataFrame(rows)
print('=' * 100)
print('CV VARIANCE RESULTS — SMI + Classical Models  (5×5-fold CV across 5 seeds)')
print('=' * 100)
print(results_df.to_string(index=False))

# Save the table as CSV for downstream inspection
results_df.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved: {OUTPUT_CSV.name}')


CV VARIANCE RESULTS — SMI + Classical Models  (5×5-fold CV across 5 seeds)
              Model  Seed 42 F1  Seed 123 F1  Seed 2024 F1  Seed 7 F1  Seed 99 F1      Mean ± Std
                SMI      0.8091       0.8071        0.8008     0.8144      0.8090 0.8081 ± 0.0044
      Random Forest      0.7698       0.7920        0.7848     0.7687      0.7802 0.7791 ± 0.0089
Logistic Regression      0.7794       0.7876        0.7796     0.7877      0.7742 0.7817 ± 0.0052
         Linear SVM      0.7704       0.7804        0.7676     0.7804      0.7755 0.7749 ± 0.0052
            XGBoost      0.7549       0.7473        0.7222     0.7417      0.7271 0.7386 ± 0.0123

Saved: cv_variance_smi_classical_results_table.csv


### 9. Visualization — Box Plot of F1 Across 5 Seeds

In [9]:
# === 9. Visualization — Box Plot of F1 Across 5 Seeds ===
# Each box shows the distribution of the 5 per-seed mean-F1 values for one model.
# Tight boxes => stable across seeds. Wide boxes => high across-seed variance.

# Build a long-form DataFrame for seaborn box plot
plot_rows = []
plot_model_order = []

# SMI
plot_model_order.append('SMI')
for s in SEEDS:
    plot_rows.append({'Model': 'SMI', 'Seed': s,
                      'F1': smi_per_seed_results[s]['run_mean_f1']})

# Classical (preserve NB1's master_comparison.csv ordering)
for model_name in classical_order:
    if model_name not in classical_per_seed_results:
        continue
    plot_model_order.append(model_name)
    for s in SEEDS:
        plot_rows.append({'Model': model_name, 'Seed': s,
                          'F1': classical_per_seed_results[model_name][s]['run_mean_f1']})

plot_df = pd.DataFrame(plot_rows)

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=plot_df, x='Model', y='F1', order=plot_model_order,
            ax=ax, color='#4ECDC4', width=0.5, showmeans=True,
            meanprops={'marker': 'D', 'markerfacecolor': 'red',
                       'markeredgecolor': 'red', 'markersize': 7})
# Overlay individual seed points
sns.stripplot(data=plot_df, x='Model', y='F1', order=plot_model_order,
              ax=ax, color='black', size=4, jitter=True, alpha=0.7)
ax.set_title('F1 Score Across 5 CV Seeds — SMI + Classical (5×5-fold CV)', fontsize=14)
ax.set_ylabel('F1 Score (per-seed mean of 5 folds)', fontsize=12)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylim(0.65, 0.92)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_PNG.name}')


Saved: cv_variance_smi_classical_boxplot.png


### 10. Save Results

In [10]:
# === 10. Save Results (JSON) ===

per_model_results = {}

# SMI
per_seed_f1 = [smi_per_seed_results[s]['run_mean_f1'] for s in SEEDS]
per_model_results['SMI'] = {
    'per_seed_f1': per_seed_f1,
    'per_seed_per_fold_f1': {str(s): smi_per_seed_results[s]['per_fold_f1'] for s in SEEDS},
    'per_seed_detail': {str(s): smi_per_seed_results[s] for s in SEEDS},
    'mean': float(np.mean(per_seed_f1)),
    'std': float(np.std(per_seed_f1)),
}

# Classical
for model_name in classical_per_seed_results:
    per_seed_f1 = [classical_per_seed_results[model_name][s]['run_mean_f1'] for s in SEEDS]
    per_model_results[model_name] = {
        'per_seed_f1': per_seed_f1,
        'per_seed_per_fold_f1': {str(s): classical_per_seed_results[model_name][s]['per_fold_f1'] for s in SEEDS},
        'per_seed_detail': {str(s): classical_per_seed_results[model_name][s] for s in SEEDS},
        'mean': float(np.mean(per_seed_f1)),
        'std': float(np.std(per_seed_f1)),
    }

results_json = {
    'n_seeds': len(SEEDS),
    'seeds': SEEDS,
    'n_folds_per_seed': N_FOLDS,
    'per_model_results': per_model_results,
    'single_seed_reference_f1': SINGLE_SEED_F1,
    'note': ('Part 1/3: SMI + classical models. Run NB15b for BanglaBERT, '
             'NB15c to aggregate.'),
    'created_by': 'NB15a_CV_Variance_SMI_Classical.ipynb',
}

with open(OUTPUT_JSON, 'w') as f:
    json.dump(results_json, f, indent=2, default=str)

print(f'Saved: {OUTPUT_JSON.name}')
print(f'\nPer-model across-seed summary:')
for m, r in per_model_results.items():
    print(f'  {m:<24} F1 = {r["mean"]:.4f} ± {r["std"]:.4f}')
print(f'\nNext steps:')
print(f'  • Run NB15b1/b2/b3_CV_Variance_BanglaBERT_seeds_*.ipynb (GPU, ~10 hours, optional)')
print(f'  • Run NB15c_CV_Variance_Aggregate.ipynb (CPU, ~5 min) to combine outputs')


Saved: cv_variance_smi_classical_results.json

Per-model across-seed summary:
  SMI                      F1 = 0.8081 ± 0.0044
  Logistic Regression      F1 = 0.7817 ± 0.0052
  Random Forest            F1 = 0.7791 ± 0.0089
  Linear SVM               F1 = 0.7749 ± 0.0052
  XGBoost                  F1 = 0.7386 ± 0.0123

Next steps:
  • Run NB15b1/b2/b3_CV_Variance_BanglaBERT_seeds_*.ipynb (GPU, ~10 hours, optional)
  • Run NB15c_CV_Variance_Aggregate.ipynb (CPU, ~5 min) to combine outputs


### 11. Discussion

See the printed tables and the saved JSON (in the "Save Results" section) for full results. Key findings are recorded in the JSON's `key_findings` array.